# Notebook 3: Input Representations and Embeddings in AlphaFold2

**Series:** Understanding AlphaFold2 — Notebook 3 of 8

**Objective:** Understand how AlphaFold2 converts raw sequence and MSA data into the tensor representations that feed the Evoformer.

---

AlphaFold2 operates on two core tensor representations: the **MSA representation** and the **pair representation**. Before any attention or geometric reasoning occurs, the model must convert biological data — amino acid sequences, multiple sequence alignments, and structural templates — into dense numerical tensors. This notebook dissects every step of that conversion with mathematical rigor and visualizations.

**Prerequisites:** Familiarity with protein sequences, MSAs (Notebook 1), and basic linear algebra.

**Dependencies:** `numpy`, `matplotlib`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import matplotlib.gridspec as gridspec

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 12})

np.random.seed(42)
print("Libraries loaded.")

---
## 1. From Sequences to Tensors

A protein sequence is a string of characters from the 20 standard amino acids. Neural networks, however, require numerical input. The first step in any protein deep learning pipeline is converting discrete residue identities into continuous vector spaces.

### One-Hot Encoding

Let the amino acid alphabet be $\mathcal{A} = \{\text{A, C, D, E, F, G, H, I, K, L, M, N, P, Q, R, S, T, V, W, Y, gap}\}$ with $|\mathcal{A}| = 21$. Each residue $s_i$ at position $i$ is assigned an index $k_i \in \{1, 2, \ldots, 21\}$. The one-hot encoding is:

$$
\mathbf{e}_i = \text{onehot}(s_i) \in \{0, 1\}^{21}, \quad (\mathbf{e}_i)_k = \begin{cases} 1 & \text{if } k = k_i \\ 0 & \text{otherwise} \end{cases}
$$

For a sequence of length $N_{\text{res}}$, the full one-hot matrix is $\mathbf{E} \in \{0,1\}^{N_{\text{res}} \times 21}$. This is a sparse, maximally non-committal representation: no amino acid is presumed similar to any other.

In [ ]:
# One-hot encoding visualization for "ACDEFG"

AA_ALPHABET = list("ACDEFGHIKLMNPQRSTVWY-")
AA_TO_IDX = {aa: i for i, aa in enumerate(AA_ALPHABET)}

def one_hot_encode(sequence, alphabet=AA_ALPHABET):
    """One-hot encode a protein sequence."""
    n = len(sequence)
    d = len(alphabet)
    encoding = np.zeros((n, d), dtype=np.float32)
    for i, residue in enumerate(sequence):
        idx = AA_TO_IDX.get(residue, len(alphabet) - 1)  # gap for unknown
        encoding[i, idx] = 1.0
    return encoding

sequence = "ACDEFG"
onehot = one_hot_encode(sequence)

fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(onehot.T, cmap='Blues', aspect='auto', vmin=0, vmax=1)

ax.set_xticks(range(len(sequence)))
ax.set_xticklabels(list(sequence), fontsize=14)
ax.set_yticks(range(len(AA_ALPHABET)))
ax.set_yticklabels(AA_ALPHABET, fontsize=11)
ax.set_xlabel('Sequence position', fontsize=13)
ax.set_ylabel('Amino acid channel', fontsize=13)
ax.set_title('One-Hot Encoding of Sequence "ACDEFG"', fontsize=14)

# Annotate the 1-values
for i in range(onehot.shape[0]):
    for j in range(onehot.shape[1]):
        if onehot[i, j] == 1.0:
            ax.text(i, j, '1', ha='center', va='center', fontsize=12, color='white')

plt.colorbar(im, ax=ax, shrink=0.8, label='Value')
plt.tight_layout()
plt.show()

print(f"Sequence: {sequence}")
print(f"One-hot shape: {onehot.shape}  (N_res x 21)")

---
## 2. The MSA Representation

The multiple sequence alignment (MSA) is one of AlphaFold2's most powerful inputs. Rather than processing a single sequence, AF2 ingests an alignment of evolutionarily related sequences, extracting co-evolutionary signals that encode structural constraints.

### Tensor Shape

The MSA representation is a 3D tensor:

$$
\mathbf{M} \in \mathbb{R}^{N_{\text{seq}} \times N_{\text{res}} \times c_m}
$$

where:
- $N_{\text{seq}}$ is the number of aligned sequences (typically $\sim 512$ after clustering),
- $N_{\text{res}}$ is the number of residue positions (the query length),
- $c_m = 256$ is the MSA channel dimension.

### Initial Embedding

Each element $\mathbf{m}_{ij} \in \mathbb{R}^{c_m}$ for sequence $i$ and residue position $j$ is initialized as:

$$
\mathbf{m}_{ij} = \text{Linear}_{\text{aa}}\bigl(\text{onehot}(s_{ij})\bigr) + \text{Linear}_{\text{del}}\bigl(\text{has\_deletion}_{ij}\bigr) + \mathbf{p}_j
$$

where:
- $\text{Linear}_{\text{aa}}: \mathbb{R}^{23} \to \mathbb{R}^{c_m}$ projects the amino acid one-hot (23 classes: 20 AAs + unknown + gap + masked),
- $\text{Linear}_{\text{del}}: \mathbb{R}^{2} \to \mathbb{R}^{c_m}$ encodes deletion information (has\_deletion, deletion\_count features),
- $\mathbf{p}_j \in \mathbb{R}^{c_m}$ is a positional embedding.

The first row ($i=0$) of the MSA always corresponds to the **query sequence** and plays a special role throughout the architecture.

In [ ]:
# Schematic 3D visualization of the MSA representation tensor

fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

# Dimensions of the rectangular prism
N_seq, N_res, c_m = 8, 12, 6  # small for visualization (represents 512, L, 256)

# Draw the rectangular prism using faces
def draw_prism(ax, origin, dx, dy, dz, color='steelblue', alpha=0.12):
    """Draw a rectangular prism."""
    x, y, z = origin
    # Define the 8 vertices
    verts = [
        [x, y, z], [x+dx, y, z], [x+dx, y+dy, z], [x, y+dy, z],
        [x, y, z+dz], [x+dx, y, z+dz], [x+dx, y+dy, z+dz], [x, y+dy, z+dz]
    ]
    # 6 faces
    faces = [
        [verts[0], verts[1], verts[5], verts[4]],  # front
        [verts[2], verts[3], verts[7], verts[6]],  # back
        [verts[0], verts[3], verts[7], verts[4]],  # left
        [verts[1], verts[2], verts[6], verts[5]],  # right
        [verts[0], verts[1], verts[2], verts[3]],  # bottom
        [verts[4], verts[5], verts[6], verts[7]],  # top
    ]
    face_colors = [
        (0.26, 0.53, 0.79, 0.15),  # front
        (0.26, 0.53, 0.79, 0.08),  # back
        (0.20, 0.45, 0.70, 0.12),  # left
        (0.35, 0.60, 0.85, 0.18),  # right
        (0.22, 0.48, 0.72, 0.10),  # bottom
        (0.40, 0.65, 0.88, 0.20),  # top
    ]
    collection = Poly3DCollection(faces, facecolors=face_colors,
                                   edgecolors='steelblue', linewidths=1.5)
    ax.add_collection3d(collection)

draw_prism(ax, (0, 0, 0), N_res, N_seq, c_m)

# Axis labels with dimension annotations
ax.set_xlabel('Residue positions ($N_{\mathrm{res}}$)', fontsize=13, labelpad=12)
ax.set_ylabel('Sequences ($N_{\mathrm{seq}}$)', fontsize=13, labelpad=12)
ax.set_zlabel('Channels ($c_m$)', fontsize=13, labelpad=10)

# Dimension annotations
ax.text(N_res/2, -2.0, -1.2, f'$N_{{\\mathrm{{res}}}}$ (e.g., 256)', fontsize=12,
        ha='center', color='darkblue')
ax.text(-2.5, N_seq/2, -1.2, f'$N_{{\\mathrm{{seq}}}}$ (e.g., 512)', fontsize=12,
        ha='center', color='darkgreen')
ax.text(-2.5, -1.5, c_m/2, f'$c_m = 256$', fontsize=12,
        ha='center', color='darkred')

# Highlight query sequence (first row)
draw_prism(ax, (0, 0, 0), N_res, 1, c_m, color='orange', alpha=0.3)
ax.text(N_res + 0.5, 0.5, c_m/2, 'Query\nsequence\n($i=0$)', fontsize=11,
        color='darkorange', ha='left', va='center')

# Mark a single element m_ij
i_mark, j_mark = 3, 5
ax.scatter([j_mark + 0.5], [i_mark + 0.5], [c_m/2], s=100, c='red', marker='o', zorder=10)
ax.text(j_mark + 1.2, i_mark + 0.5, c_m/2 + 0.8,
        '$\\mathbf{m}_{ij} \\in \\mathbb{R}^{c_m}$', fontsize=12, color='red')

ax.set_xlim([-2, N_res + 4])
ax.set_ylim([-2, N_seq + 2])
ax.set_zlim([-1, c_m + 2])
ax.set_title('MSA Representation Tensor: $\\mathbf{M} \\in \\mathbb{R}^{N_{\\mathrm{seq}} \\times N_{\\mathrm{res}} \\times c_m}$',
             fontsize=14, pad=15)
ax.view_init(elev=20, azim=-55)
ax.set_xticks([])
ax.set_yticks([])
ax.set_zticks([])

plt.tight_layout()
plt.show()

---
## 3. The Pair Representation

While the MSA representation captures per-sequence, per-residue features, the **pair representation** captures relationships between every pair of residue positions. This is the key data structure for encoding spatial proximity, co-evolutionary coupling, and geometric constraints.

### Tensor Shape

$$
\mathbf{Z} \in \mathbb{R}^{N_{\text{res}} \times N_{\text{res}} \times c_z}, \quad c_z = 128
$$

The element $\mathbf{z}_{ij} \in \mathbb{R}^{c_z}$ encodes all learned features about the relationship between residue $i$ and residue $j$.

### Initialization

The pair representation is initialized from three sources:

1. **Relative positional encoding:** Captures the sequence separation between residues.
2. **Template features:** Structural information from homologous templates.
3. **Outer product of MSA features:** Converts MSA column correlations into pairwise features.

### Relative Positional Encoding

The relative position between residues $i$ and $j$ is:

$$
d_{ij} = \text{clip}(j - i, -32, 32)
$$

This is then one-hot encoded into 65 bins (values $-32, -31, \ldots, 0, \ldots, 31, 32$) and linearly projected:

$$
\mathbf{z}_{ij}^{\text{pos}} = \text{Linear}\bigl(\text{onehot}(d_{ij})\bigr) \in \mathbb{R}^{c_z}
$$

This encoding is **translation-invariant** (depends only on $j - i$, not absolute positions) and **bounded** (residues far apart in sequence are grouped into the same bin).

In [ ]:
# Relative positional encoding visualization

N = 50  # number of residues

# Compute clipped relative positions
positions = np.arange(N)
rel_pos = positions[None, :] - positions[:, None]  # shape (N, N)
rel_pos_clipped = np.clip(rel_pos, -32, 32)

# One-hot expansion: 65 bins for values -32..+32
n_bins = 65
bin_values = np.arange(-32, 33)  # -32 to +32
onehot_relpos = np.zeros((N, N, n_bins), dtype=np.float32)
for b, val in enumerate(bin_values):
    onehot_relpos[:, :, b] = (rel_pos_clipped == val).astype(np.float32)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: raw clipped relative positions
im0 = axes[0].imshow(rel_pos_clipped, cmap='RdBu_r', aspect='equal',
                       vmin=-32, vmax=32)
axes[0].set_xlabel('Residue $j$', fontsize=13)
axes[0].set_ylabel('Residue $i$', fontsize=13)
axes[0].set_title('Relative Position: $\\mathrm{clip}(j - i, -32, 32)$', fontsize=14)
plt.colorbar(im0, ax=axes[0], shrink=0.8, label='Clipped offset')

# Right: one-hot expansion for selected residue pairs
# Show slices at i=0, i=10, i=25, i=40
selected_i = [0, 10, 25, 40]
j_fixed = 25
onehot_slices = np.zeros((len(selected_i), n_bins))
for k, i in enumerate(selected_i):
    onehot_slices[k] = onehot_relpos[i, j_fixed, :]

im1 = axes[1].imshow(onehot_slices, cmap='Blues', aspect='auto')
axes[1].set_yticks(range(len(selected_i)))
axes[1].set_yticklabels([f'$i={i}, j={j_fixed}$' for i in selected_i], fontsize=11)
axes[1].set_xlabel('Bin index (encoding $-32$ to $+32$)', fontsize=13)
axes[1].set_title('One-Hot Encoding of Relative Position (column $j=25$)', fontsize=14)
plt.colorbar(im1, ax=axes[1], shrink=0.8)

# Mark the active bins
for k, i in enumerate(selected_i):
    active_bin = int(np.argmax(onehot_slices[k]))
    axes[1].plot(active_bin, k, 'rv', markersize=8)

plt.tight_layout()
plt.show()

print(f"Relative position matrix shape: {rel_pos_clipped.shape}")
print(f"One-hot expansion shape: {onehot_relpos.shape} (N_res x N_res x 65 bins)")
print(f"After linear projection: ({N}, {N}, c_z=128)")

---
## 4. Positional Encoding Deep Dive

Positional encoding is fundamental to any transformer architecture: without it, the model is permutation-invariant and cannot distinguish position 5 from position 50. AlphaFold2's choice of **relative** positional encoding differs significantly from the original transformer's **absolute sinusoidal** encoding.

### Sinusoidal Positional Encoding (Original Transformer)

The original transformer (Vaswani et al., 2017) uses:

$$
PE(\text{pos}, 2k) = \sin\!\left(\frac{\text{pos}}{10000^{2k/d_{\text{model}}}}\right), \quad
PE(\text{pos}, 2k+1) = \cos\!\left(\frac{\text{pos}}{10000^{2k/d_{\text{model}}}}\right)
$$

where $\text{pos}$ is the absolute position and $k$ indexes the embedding dimension. This produces a unique, smooth fingerprint for each position, and the dot product between two position encodings depends on their relative offset.

### Relative Positional Encoding (AlphaFold2)

AlphaFold2 instead directly encodes the **offset** $j - i$ between residue pairs:

$$
\mathbf{z}_{ij}^{\text{pos}} = \text{Linear}\bigl(\text{onehot}(\text{clip}(j - i, -32, 32))\bigr)
$$

**Why relative?** Protein structure is governed by local interactions (secondary structure within $\sim 5{-}10$ residues) and contacts that span many residues. The relative encoding:
1. Is exactly **translation-invariant**: shifting the entire sequence does not change pairwise features.
2. Provides **fine-grained** resolution for nearby residues ($|j-i| \leq 32$).
3. Groups **distant** pairs into a single bin, reflecting the biological insight that exact long-range sequence separation matters less than proximity in 3D.

In [ ]:
# Side-by-side comparison: sinusoidal vs. relative positional encoding

N = 50
d_model = 64  # embedding dimension for sinusoidal

# --- Sinusoidal positional encoding ---
sinusoidal_pe = np.zeros((N, d_model))
for pos in range(N):
    for k in range(d_model // 2):
        angle = pos / (10000 ** (2 * k / d_model))
        sinusoidal_pe[pos, 2*k] = np.sin(angle)
        sinusoidal_pe[pos, 2*k+1] = np.cos(angle)

# Pairwise similarity matrix from sinusoidal encoding
# Normalize rows and compute dot products
norms = np.linalg.norm(sinusoidal_pe, axis=1, keepdims=True)
sinusoidal_normed = sinusoidal_pe / norms
sinusoidal_sim = sinusoidal_normed @ sinusoidal_normed.T

# --- Relative positional encoding ---
positions = np.arange(N)
rel_pos_clipped = np.clip(positions[None, :] - positions[:, None], -32, 32)

# Simulate a learned linear projection from one-hot(65) to c_z=64
# For visualization, we just show the raw clipped values as a comparison

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: sinusoidal positional encoding
im0 = axes[0].imshow(sinusoidal_pe.T, cmap='RdBu_r', aspect='auto',
                       vmin=-1, vmax=1)
axes[0].set_xlabel('Position', fontsize=13)
axes[0].set_ylabel('Encoding dimension', fontsize=13)
axes[0].set_title('Sinusoidal Positional Encoding\n(Absolute, NLP Transformer)', fontsize=14)
plt.colorbar(im0, ax=axes[0], shrink=0.8)

# Right: relative positional encoding (clipped offsets)
im1 = axes[1].imshow(rel_pos_clipped, cmap='RdBu_r', aspect='equal',
                       vmin=-32, vmax=32)
axes[1].set_xlabel('Residue $j$', fontsize=13)
axes[1].set_ylabel('Residue $i$', fontsize=13)
axes[1].set_title('Relative Positional Encoding\n(Pairwise, AlphaFold2)', fontsize=14)
plt.colorbar(im1, ax=axes[1], shrink=0.8, label='clip$(j-i, -32, 32)$')

plt.tight_layout()
plt.show()

# Additional comparison: pairwise dot-product similarity
fig2, axes2 = plt.subplots(1, 2, figsize=(14, 5.5))

im2 = axes2[0].imshow(sinusoidal_sim, cmap='viridis', aspect='equal')
axes2[0].set_xlabel('Position $j$', fontsize=13)
axes2[0].set_ylabel('Position $i$', fontsize=13)
axes2[0].set_title('Sinusoidal: Pairwise Dot-Product Similarity', fontsize=13)
plt.colorbar(im2, ax=axes2[0], shrink=0.8)

# For relative encoding, pairwise "similarity" is just whether offsets match
# Visualize the effective information: after linear projection, 
# distinct offsets get distinct embeddings
# Show the number of distinct offsets per row
offset_identity = (rel_pos_clipped[:, :, None] == rel_pos_clipped[:, None, :]).astype(float).mean(axis=-1)
im3 = axes2[1].imshow(rel_pos_clipped, cmap='coolwarm', aspect='equal', vmin=-32, vmax=32)
axes2[1].set_xlabel('Position $j$', fontsize=13)
axes2[1].set_ylabel('Position $i$', fontsize=13)
axes2[1].set_title('Relative: Clipped Offset Matrix (feeds into pair repr.)', fontsize=13)
plt.colorbar(im3, ax=axes2[1], shrink=0.8)

plt.tight_layout()
plt.show()

print("Key difference: Sinusoidal encoding is absolute (per-position vector),")
print("while AF2 relative encoding is pairwise (per-pair feature).")
print(f"Sinusoidal PE shape: ({N}, {d_model}) -- one vector per position")
print(f"Relative PE shape: ({N}, {N}, 65) after one-hot -- one vector per pair")

---
## 5. Template Embedding

When homologous structures are available in the PDB, AlphaFold2 can leverage them as **templates**. Template information is injected primarily into the pair representation.

### Template Features

For each template $t$ and each residue pair $(i, j)$, AF2 extracts:

- **Pseudo-beta ($C_\beta$) distances:** $d_{ij}^{(t)} = \|\mathbf{CB}_i^{(t)} - \mathbf{CB}_j^{(t)}\|$
- **Unit vectors:** $\hat{\mathbf{u}}_{ij}^{(t)} = \frac{\mathbf{CB}_j^{(t)} - \mathbf{CB}_i^{(t)}}{d_{ij}^{(t)}}$ expressed in each residue's local frame
- **Backbone torsion angles** and other geometric features

### Pseudo-Beta Approximation

The $C_\beta$ position is approximated from the three backbone atoms $\mathbf{N}_i, \mathbf{C\alpha}_i, \mathbf{C}_i$:

$$
\mathbf{b} = \mathbf{C\alpha}_i - \mathbf{N}_i, \quad \mathbf{c} = \mathbf{C}_i - \mathbf{C\alpha}_i
$$
$$
\mathbf{a} = \mathbf{b} \times \mathbf{c}
$$
$$
\mathbf{CB}_i = -0.58273431 \, \hat{\mathbf{a}} + 0.56802827 \, \hat{\mathbf{b}} - 0.54067466 \, \hat{\mathbf{c}} + \mathbf{C\alpha}_i
$$

where $\hat{\mathbf{v}} = \mathbf{v} / \|\mathbf{v}\|$. For glycine (which lacks a $C_\beta$), the $C_\alpha$ position is used instead.

### Template Pair Stack

Template pair features have shape $(N_{\text{templ}}, N_{\text{res}}, N_{\text{res}}, c)$ and are processed through a small transformer stack (the **template pair stack**), then reduced across templates via attention and added to the pair representation.

In [ ]:
# Synthetic template features for a 40-residue protein

N_templ_res = 40

# Generate synthetic backbone coordinates
# Simulate a helix-turn-helix motif
def generate_helix_coords(n_residues, start=np.array([0, 0, 0])):
    """Generate synthetic CA coordinates along a helix."""
    coords = np.zeros((n_residues, 3))
    # Helix parameters: rise = 1.5 A, radius = 2.3 A, ~100 deg per residue
    for i in range(n_residues):
        angle = np.radians(100 * i)
        if i < 15:  # first helix
            coords[i] = [2.3 * np.cos(angle), 2.3 * np.sin(angle), 1.5 * i]
        elif i < 20:  # turn
            coords[i] = [2.3 * np.cos(angle) + 3 * (i - 15), 
                         2.3 * np.sin(angle), 1.5 * 15 + 0.5 * (i - 15)]
        else:  # second helix going back
            angle2 = np.radians(100 * (i - 20) + 180)
            coords[i] = [2.3 * np.cos(angle2) + 15, 
                         2.3 * np.sin(angle2), 
                         1.5 * 15 + 2.5 - 1.5 * (i - 20)]
    return coords + start

ca_coords = generate_helix_coords(N_templ_res)

# Compute CB distances (using CA as proxy)
# Add small perturbation to simulate CB offset
cb_coords = ca_coords + np.random.randn(N_templ_res, 3) * 0.5

# Distance matrix
diff = cb_coords[:, None, :] - cb_coords[None, :, :]
dist_matrix = np.sqrt((diff ** 2).sum(axis=-1))

# Unit vector field
unit_vectors = np.zeros_like(diff)
safe_dist = np.maximum(dist_matrix[:, :, None], 1e-8)
unit_vectors = diff / safe_dist

fig, axes = plt.subplots(1, 2, figsize=(16, 6.5))

# Left: template distance matrix
im0 = axes[0].imshow(dist_matrix, cmap='magma_r', aspect='equal')
axes[0].set_xlabel('Residue $j$', fontsize=13)
axes[0].set_ylabel('Residue $i$', fontsize=13)
axes[0].set_title('Template $C_\\beta$ Distance Matrix ($d_{ij}$)', fontsize=14)
cbar0 = plt.colorbar(im0, ax=axes[0], shrink=0.8)
cbar0.set_label('Distance (\u00c5)', fontsize=12)

# Mark contact threshold
contact_mask = dist_matrix < 8.0
axes[0].contour(contact_mask.astype(float), levels=[0.5], colors='cyan',
                linewidths=0.8, linestyles='--')
axes[0].text(N_templ_res - 2, 3, '$< 8$ \u00c5', fontsize=11,
             color='cyan', ha='right')

# Right: unit vector field (subsample for clarity)
step = 3
i_idx, j_idx = np.meshgrid(np.arange(0, N_templ_res, step),
                            np.arange(0, N_templ_res, step), indexing='ij')
# Show x and y components of unit vectors
U = unit_vectors[i_idx, j_idx, 0]  # x-component
V = unit_vectors[i_idx, j_idx, 1]  # y-component

# Color by distance
colors = dist_matrix[i_idx, j_idx]

q = axes[1].quiver(j_idx, i_idx, U, V, colors, cmap='coolwarm',
                    scale=25, width=0.003, alpha=0.8)
axes[1].set_xlabel('Residue $j$', fontsize=13)
axes[1].set_ylabel('Residue $i$', fontsize=13)
axes[1].set_title('Template Unit Vector Field ($\\hat{u}_{ij}$, $xy$-components)', fontsize=14)
axes[1].set_aspect('equal')
axes[1].set_xlim(-1, N_templ_res)
axes[1].set_ylim(N_templ_res, -1)
cbar1 = plt.colorbar(q, ax=axes[1], shrink=0.8)
cbar1.set_label('Distance (\u00c5)', fontsize=12)

plt.tight_layout()
plt.show()

print(f"Template distance matrix shape: ({N_templ_res}, {N_templ_res})")
print(f"Template unit vectors shape: ({N_templ_res}, {N_templ_res}, 3)")
print(f"Full template pair features: (N_templ, {N_templ_res}, {N_templ_res}, c)")

---
## 6. Extra MSA Features

AlphaFold2 often retrieves very large MSAs (up to $\sim$100,000 sequences from databases like BFD, MGnify, and Uniclust30). Processing all of these through the full Evoformer would be prohibitively expensive. AF2 therefore partitions the MSA into two groups:

### Cluster MSA

The raw MSA is first clustered (e.g., via a greedy cover algorithm at 50% sequence identity). The cluster centers form the **cluster MSA**, typically $\sim 512$ sequences. These feed into the main Evoformer.

For each cluster center, a **cluster profile** is computed by averaging the one-hot encodings of all sequences within that cluster, providing a soft summary of sequence diversity.

### Extra MSA Stack

The remaining sequences (up to $\sim 5{,}000$) form the **extra MSA**. These are processed through a lighter-weight **extra MSA stack** consisting of 4 blocks (vs. 48 for the Evoformer). The extra MSA stack uses:

- Global column-wise attention (instead of full attention)
- Pair-biased row-wise attention
- Outer product mean, which **updates the pair representation**

This design extracts the co-evolutionary signal from thousands of sequences at moderate computational cost.

In [ ]:
# Data flow diagram: Raw MSA --> Cluster MSA + Extra MSA --> Evoformer

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_aspect('equal')

def draw_box(ax, xy, w, h, text, color='lightsteelblue', textcolor='black', fontsize=12):
    """Draw a labeled box."""
    box = FancyBboxPatch(xy, w, h, boxstyle='round,pad=0.15',
                         facecolor=color, edgecolor='gray', linewidth=1.5)
    ax.add_patch(box)
    cx, cy = xy[0] + w/2, xy[1] + h/2
    ax.text(cx, cy, text, ha='center', va='center',
            fontsize=fontsize, color=textcolor, wrap=True)

def draw_arrow(ax, start, end, color='gray'):
    """Draw an arrow between two points."""
    arrow = FancyArrowPatch(start, end, arrowstyle='-|>',
                            mutation_scale=18, color=color, linewidth=2)
    ax.add_patch(arrow)

# Raw MSA
draw_box(ax, (0.5, 7.5), 3.0, 1.5, 'Raw MSA\n(~100k seqs)', 
         color='#FFD580', fontsize=13)

# Clustering step
draw_arrow(ax, (2.0, 7.5), (2.0, 6.8), color='dimgray')
ax.text(2.5, 7.1, 'Cluster at\n50% identity', fontsize=11, color='dimgray', va='center')

# Cluster MSA
draw_box(ax, (0.3, 4.8), 2.5, 1.8, 'Cluster MSA\n(~512 seqs)\n+ profiles',
         color='#87CEEB', fontsize=12)

# Extra MSA
draw_box(ax, (3.5, 4.8), 2.5, 1.8, 'Extra MSA\n(~5000 seqs)',
         color='#98FB98', fontsize=12)

# Arrows from raw to cluster and extra
draw_arrow(ax, (1.5, 7.5), (1.5, 6.6))
draw_arrow(ax, (2.5, 7.5), (4.75, 6.6))

# Evoformer
draw_box(ax, (7.5, 4.0), 3.0, 2.5, 'Evoformer\n(48 blocks)\n\nMSA repr +\nPair repr',
         color='#DDA0DD', fontsize=13)

# Extra MSA Stack
draw_box(ax, (3.5, 2.0), 2.5, 1.8, 'Extra MSA\nStack\n(4 blocks)',
         color='#90EE90', fontsize=12)

# Pair representation
draw_box(ax, (7.8, 1.0), 2.5, 1.5, 'Pair\nRepresentation',
         color='#FFB6C1', fontsize=12)

# Arrows
draw_arrow(ax, (2.8, 5.7), (7.5, 5.7))   # Cluster MSA --> Evoformer
draw_arrow(ax, (4.75, 4.8), (4.75, 3.8))  # Extra MSA --> Extra MSA Stack
draw_arrow(ax, (6.0, 2.9), (7.8, 1.8))    # Extra MSA Stack --> Pair repr
draw_arrow(ax, (9.0, 4.0), (9.0, 2.5))    # Evoformer --> Pair repr (bidirectional)
draw_arrow(ax, (9.3, 2.5), (9.3, 4.0))    # Pair repr --> Evoformer

# Labels on arrows
ax.text(5.0, 6.0, 'MSA embedding', fontsize=11, color='navy', style='italic')
ax.text(5.3, 3.3, 'Outer product\nmean', fontsize=11, color='darkgreen', style='italic')
ax.text(9.5, 3.3, 'Updates', fontsize=11, color='purple', style='italic')

# Title and dimension annotations
ax.text(7.0, 9.3, 'AlphaFold2 MSA Processing Pipeline', fontsize=16,
        ha='center', color='black')

# Dimension annotations
ax.text(0.3, 4.55, '$(512, L, 256)$', fontsize=10, color='steelblue')
ax.text(3.5, 4.55, '$(5000, L, 64)$', fontsize=10, color='green')
ax.text(7.8, 0.75, '$(L, L, 128)$', fontsize=10, color='crimson')

plt.tight_layout()
plt.show()

---
## 7. Recycling: Iterative Refinement

One of AlphaFold2's key design choices is **recycling**: running the entire network multiple times (typically 3 iterations), feeding outputs from one pass back as additional inputs to the next.

### Recycled Features

At each recycling iteration $r \to r+1$, two quantities are fed back:

1. **First-row MSA representation** $\mathbf{m}_{1j}^{(r)}$: the query sequence's representation from the previous pass.
2. **Pair representation** $\mathbf{z}_{ij}^{(r)}$: the full pairwise feature tensor.

These are incorporated via:

$$
\mathbf{m}_{ij}^{(r+1)} = f_{\text{embed}}(\text{input features}) + \begin{cases} \text{Linear}(\text{LayerNorm}(\mathbf{m}_{1j}^{(r)})) & \text{if } i = 1 \\ \mathbf{0} & \text{otherwise} \end{cases}
$$

$$
\mathbf{z}_{ij}^{(r+1)} = g_{\text{embed}}(\text{input features}) + \text{Linear}(\text{LayerNorm}(\mathbf{z}_{ij}^{(r)}))
$$

At the first iteration ($r=0$), the recycled contributions are zero. Importantly, **gradients are not propagated through recycling iterations** during training; each pass is treated independently, and the loss is computed only on the final iteration's output.

### Why Recycling Works

Recycling provides a form of iterative refinement akin to an unrolled optimization. The structure module's output from iteration $r$ implicitly encodes a 3D structure hypothesis, and feeding the corresponding representations back allows the Evoformer to refine its understanding conditioned on that hypothesis.

In [ ]:
# Simulate recycling: model confidence improving over iterations

n_recycles = 3
n_residues = 80

# Simulate pLDDT (predicted local distance difference test) improving over iterations
# pLDDT ranges from 0 to 100; higher is better
np.random.seed(42)

# Base confidence pattern with secondary structure signal
base_confidence = np.zeros(n_residues)
# Helical regions: higher confidence
helix1 = slice(5, 22)
helix2 = slice(30, 48)
helix3 = slice(55, 72)
loop1 = slice(22, 30)
loop2 = slice(48, 55)

base_confidence[helix1] = 85
base_confidence[helix2] = 88
base_confidence[helix3] = 82
base_confidence[loop1] = 60
base_confidence[loop2] = 55
base_confidence[:5] = 45   # N-terminus (disordered)
base_confidence[72:] = 40  # C-terminus (disordered)

# Each recycling iteration improves confidence
recycling_data = []
scales = [0.55, 0.80, 1.0]  # fraction of final confidence at each iteration
noise_scales = [8, 4, 2]     # noise decreases with iterations

for r in range(n_recycles):
    confidence = base_confidence * scales[r] + np.random.randn(n_residues) * noise_scales[r]
    confidence = np.clip(confidence, 10, 100)
    recycling_data.append(confidence)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: per-residue confidence across iterations
colors_iter = ['#E74C3C', '#F39C12', '#27AE60']
for r in range(n_recycles):
    axes[0].plot(range(n_residues), recycling_data[r], 
                 color=colors_iter[r], linewidth=2,
                 label=f'Iteration {r+1}', alpha=0.85)

# Shade secondary structure regions
for s, label in [(helix1, 'Helix'), (helix2, 'Helix'), (helix3, 'Helix')]:
    axes[0].axvspan(s.start, s.stop, alpha=0.08, color='blue')
for s in [loop1, loop2]:
    axes[0].axvspan(s.start, s.stop, alpha=0.08, color='orange')

axes[0].set_xlabel('Residue index', fontsize=13)
axes[0].set_ylabel('pLDDT (confidence)', fontsize=13)
axes[0].set_title('Per-Residue Confidence Across Recycling Iterations', fontsize=14)
axes[0].legend(fontsize=12)
axes[0].set_ylim(0, 105)
axes[0].axhline(y=70, color='gray', linestyle='--', alpha=0.5)
axes[0].text(n_residues - 1, 72, 'Confident threshold', fontsize=10,
             ha='right', color='gray')

# Right: aggregate metrics over iterations
mean_plddt = [np.mean(d) for d in recycling_data]
median_plddt = [np.median(d) for d in recycling_data]
plddt_70_frac = [np.mean(d > 70) * 100 for d in recycling_data]

iterations = [1, 2, 3]
ax2 = axes[1]
ax2.plot(iterations, mean_plddt, 'o-', color='#2980B9', linewidth=2.5,
         markersize=10, label='Mean pLDDT')
ax2.plot(iterations, median_plddt, 's-', color='#E67E22', linewidth=2.5,
         markersize=10, label='Median pLDDT')

ax2_twin = ax2.twinx()
ax2_twin.bar(iterations, plddt_70_frac, alpha=0.2, color='green', width=0.3,
             label='% residues > 70')
ax2_twin.set_ylabel('% Residues with pLDDT > 70', fontsize=12, color='green')
ax2_twin.set_ylim(0, 100)
ax2_twin.tick_params(axis='y', labelcolor='green')

ax2.set_xlabel('Recycling iteration', fontsize=13)
ax2.set_ylabel('pLDDT score', fontsize=13)
ax2.set_title('Convergence of Model Confidence with Recycling', fontsize=14)
ax2.set_xticks([1, 2, 3])
ax2.set_ylim(30, 90)
ax2.legend(loc='lower right', fontsize=12)

plt.tight_layout()
plt.show()

print("Recycling summary:")
for r in range(n_recycles):
    print(f"  Iteration {r+1}: mean pLDDT = {mean_plddt[r]:.1f}, "
          f"median = {median_plddt[r]:.1f}, "
          f"% > 70 = {plddt_70_frac[r]:.1f}%")

---
## 8. Putting It All Together: The Input Pipeline

We now have all the pieces to understand the complete input pipeline of AlphaFold2. The following diagram shows how raw biological data is transformed step by step into the two core tensor representations.

### Summary of the Pipeline

1. **Sequence** $\to$ one-hot encoding $\to$ linear projection $\to$ MSA representation (query row)
2. **MSA sequences** $\to$ one-hot + deletion features $\to$ linear projection + positional encoding $\to$ MSA representation
3. **Relative positions** $\to$ one-hot of clipped offset $\to$ linear projection $\to$ pair representation
4. **Templates** $\to$ template pair stack $\to$ attention-weighted sum $\to$ added to pair representation
5. **Extra MSA** $\to$ extra MSA stack $\to$ outer product mean $\to$ updates pair representation
6. **Recycling** $\to$ previous $\mathbf{m}_{1j}, \mathbf{z}_{ij}$ $\to$ linear projection $\to$ added to both representations

Both representations then enter the **Evoformer**, which is the subject of the next notebook.

In [ ]:
# Full input pipeline diagram

fig, ax = plt.subplots(figsize=(18, 12))
ax.set_xlim(0, 18)
ax.set_ylim(0, 13)
ax.axis('off')

def box(ax, x, y, w, h, text, color='#D6EAF8', ec='#2C3E50', fontsize=11, textcolor='black'):
    """Draw a rounded box with centered text."""
    b = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.12',
                       facecolor=color, edgecolor=ec, linewidth=1.5)
    ax.add_patch(b)
    lines = text.split('\n')
    n_lines = len(lines)
    for i, line in enumerate(lines):
        offset = (i - (n_lines - 1) / 2) * 0.32
        ax.text(x + w/2, y + h/2 - offset, line, ha='center', va='center',
                fontsize=fontsize, color=textcolor)

def arrow(ax, x1, y1, x2, y2, color='#555555'):
    a = FancyArrowPatch((x1, y1), (x2, y2), arrowstyle='-|>',
                        mutation_scale=16, color=color, linewidth=1.8)
    ax.add_patch(a)

# ====== INPUT SOURCES (top) ======
# Sequence
box(ax, 0.3, 11.2, 2.4, 1.0, 'Amino Acid\nSequence', color='#FDEBD0', fontsize=12)

# MSA
box(ax, 3.5, 11.2, 2.4, 1.0, 'MSA\n(~100k seqs)', color='#FDEBD0', fontsize=12)

# Templates
box(ax, 6.7, 11.2, 2.4, 1.0, 'Template\nStructures', color='#FDEBD0', fontsize=12)

# Recycled features
box(ax, 13.0, 11.2, 2.8, 1.0, 'Recycled Features\n($\mathbf{m}_{1j}^{(r)}, \mathbf{z}_{ij}^{(r)}$)',
    color='#E8DAEF', fontsize=11)

# ====== PROCESSING STEPS (middle) ======
# One-hot encoding
box(ax, 0.3, 9.3, 2.4, 0.9, 'One-Hot\nEncoding', color='#D5F5E3', fontsize=11)
arrow(ax, 1.5, 11.2, 1.5, 10.2)

# MSA clustering
box(ax, 3.0, 9.3, 1.5, 0.9, 'Cluster\nMSA', color='#D5F5E3', fontsize=10)
box(ax, 4.7, 9.3, 1.5, 0.9, 'Extra\nMSA', color='#D5F5E3', fontsize=10)
arrow(ax, 4.7, 11.2, 3.75, 10.2)
arrow(ax, 4.7, 11.2, 5.45, 10.2)

# Template features
box(ax, 6.7, 9.3, 2.4, 0.9, 'Template Pair\nFeatures', color='#D5F5E3', fontsize=11)
arrow(ax, 7.9, 11.2, 7.9, 10.2)

# Recycling linear projection
box(ax, 13.0, 9.3, 2.8, 0.9, 'LayerNorm +\nLinear', color='#D5F5E3', fontsize=11)
arrow(ax, 14.4, 11.2, 14.4, 10.2)

# ====== LINEAR PROJECTIONS ======
# Sequence projection
box(ax, 0.3, 7.5, 2.4, 0.9, 'Linear\nProjection', color='#AED6F1', fontsize=11)
arrow(ax, 1.5, 9.3, 1.5, 8.4)

# Cluster MSA embedding
box(ax, 3.0, 7.5, 1.5, 0.9, 'MSA\nEmbed', color='#AED6F1', fontsize=10)
arrow(ax, 3.75, 9.3, 3.75, 8.4)

# Extra MSA Stack
box(ax, 4.7, 7.5, 1.5, 0.9, 'Extra MSA\nStack', color='#AED6F1', fontsize=10)
arrow(ax, 5.45, 9.3, 5.45, 8.4)

# Template pair stack
box(ax, 6.7, 7.5, 2.4, 0.9, 'Template\nPair Stack', color='#AED6F1', fontsize=11)
arrow(ax, 7.9, 9.3, 7.9, 8.4)

# Relative position encoding
box(ax, 10.0, 9.3, 2.4, 0.9, 'Relative Pos.\nEncoding', color='#D5F5E3', fontsize=11)
box(ax, 10.0, 7.5, 2.4, 0.9, 'Linear\nProjection', color='#AED6F1', fontsize=11)
arrow(ax, 11.2, 9.3, 11.2, 8.4)

# ====== CORE REPRESENTATIONS ======
# MSA Representation
box(ax, 0.5, 4.8, 4.5, 1.8, 'MSA Representation\n$\mathbf{M} \in \mathbb{R}^{N_{seq} \\times N_{res} \\times 256}$',
    color='#85C1E9', ec='#2471A3', fontsize=13, textcolor='#1A5276')

# Pair Representation
box(ax, 6.5, 4.8, 5.5, 1.8, 'Pair Representation\n$\mathbf{Z} \in \mathbb{R}^{N_{res} \\times N_{res} \\times 128}$',
    color='#F1948A', ec='#C0392B', fontsize=13, textcolor='#7B241C')

# Arrows into representations
arrow(ax, 1.5, 7.5, 2.0, 6.6)   # Sequence linear --> MSA repr
arrow(ax, 3.75, 7.5, 3.0, 6.6)  # Cluster MSA embed --> MSA repr

arrow(ax, 5.45, 7.5, 8.0, 6.6)  # Extra MSA Stack --> Pair repr
arrow(ax, 7.9, 7.5, 8.5, 6.6)   # Template pair stack --> Pair repr
arrow(ax, 11.2, 7.5, 9.5, 6.6)  # Rel pos --> Pair repr

# Recycling arrows
arrow(ax, 13.5, 9.3, 2.75, 6.6)  # Recycled --> MSA repr
arrow(ax, 15.0, 9.3, 11.0, 6.6)  # Recycled --> Pair repr

# ====== EVOFORMER (bottom) ======
box(ax, 3.5, 1.5, 7.0, 2.2, 'Evoformer\n(48 blocks of attention + message passing)',
    color='#D2B4DE', ec='#6C3483', fontsize=14, textcolor='#4A235A')

# Arrows from representations to Evoformer
arrow(ax, 2.75, 4.8, 5.5, 3.7)
arrow(ax, 9.25, 4.8, 8.5, 3.7)

# Bidirectional arrow inside Evoformer
ax.annotate('', xy=(8.2, 2.6), xytext=(5.8, 2.6),
            arrowprops=dict(arrowstyle='<->', color='#4A235A', lw=2))
ax.text(7.0, 2.1, 'cross-talk', fontsize=11, ha='center', color='#4A235A',
        style='italic')

# Output arrow
arrow(ax, 7.0, 1.5, 7.0, 0.5)
ax.text(7.5, 0.8, 'To Structure Module', fontsize=12, color='#2C3E50', style='italic')

# Title
ax.text(9.0, 12.7, 'AlphaFold2 Input Pipeline: From Raw Data to Tensor Representations',
        fontsize=16, ha='center', color='#2C3E50')

plt.tight_layout()
plt.show()

---
## 9. Summary and Key Takeaways

This notebook has detailed the complete input pipeline of AlphaFold2, showing how raw biological data is converted into the two tensor representations that drive all downstream computation.

### Core Representations

| Representation | Shape | Content | Role |
|:---|:---|:---|:---|
| **MSA** $\mathbf{M}$ | $(N_{\text{seq}}, N_{\text{res}}, 256)$ | Per-sequence, per-residue features | Captures evolutionary variation, co-evolution |
| **Pair** $\mathbf{Z}$ | $(N_{\text{res}}, N_{\text{res}}, 128)$ | Per-residue-pair features | Encodes spatial relationships, contacts |

### Key Design Decisions

1. **Dual-track architecture:** Rather than flattening everything into a single sequence, AF2 maintains two separate but communicating representations. The MSA track captures sequence-level patterns; the pair track captures structural geometry.

2. **Relative positional encoding:** Unlike NLP transformers, AF2 uses translation-invariant pairwise offsets rather than absolute position encodings. This aligns with the physics of protein folding: structure depends on residue separations, not absolute numbering.

3. **MSA partitioning:** Processing a massive MSA in two streams (cluster + extra) balances information extraction against computational cost.

4. **Template injection:** Structural templates are processed and injected into the pair representation, providing a "warm start" for proteins with known homologs.

5. **Recycling:** Running the network multiple times with feedback produces progressively refined representations, analogous to iterative optimization.

### What Comes Next

In **Notebook 4**, we will dive into the **Evoformer**, AlphaFold2's central engine, and examine:
- Row-wise gated self-attention with pair bias
- Column-wise gated self-attention
- Outer product mean (MSA $\to$ pair)
- Triangular multiplicative updates and triangle attention (pair $\to$ pair)

These attention mechanisms operate on the representations constructed here, progressively enriching them with structural information.

In [ ]:
# Summary visualization: dimensionality overview

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: MSA representation dimensions
msa_dims = {'Sequences\n($N_{seq}$)': 512, 'Residues\n($N_{res}$)': 256, 
            'Channels\n($c_m$)': 256}
bars1 = axes[0].bar(msa_dims.keys(), msa_dims.values(), color='#85C1E9', 
                     edgecolor='#2471A3', linewidth=1.5)
for bar_item, val in zip(bars1, msa_dims.values()):
    axes[0].text(bar_item.get_x() + bar_item.get_width()/2, bar_item.get_height() + 10,
                 str(val), ha='center', fontsize=12, color='#1A5276')
axes[0].set_ylabel('Dimension size', fontsize=12)
axes[0].set_title('MSA Representation\n$\mathbf{M} \in \mathbb{R}^{512 \\times 256 \\times 256}$',
                   fontsize=13, color='#2471A3')
axes[0].set_ylim(0, 600)

# Panel 2: Pair representation dimensions
pair_dims = {'Residues $i$\n($N_{res}$)': 256, 'Residues $j$\n($N_{res}$)': 256,
             'Channels\n($c_z$)': 128}
bars2 = axes[1].bar(pair_dims.keys(), pair_dims.values(), color='#F1948A',
                     edgecolor='#C0392B', linewidth=1.5)
for bar_item, val in zip(bars2, pair_dims.values()):
    axes[1].text(bar_item.get_x() + bar_item.get_width()/2, bar_item.get_height() + 10,
                 str(val), ha='center', fontsize=12, color='#7B241C')
axes[1].set_ylabel('Dimension size', fontsize=12)
axes[1].set_title('Pair Representation\n$\mathbf{Z} \in \mathbb{R}^{256 \\times 256 \\times 128}$',
                   fontsize=13, color='#C0392B')
axes[1].set_ylim(0, 350)

# Panel 3: Memory comparison (approximate)
# Assuming float32 (4 bytes)
msa_memory = 512 * 256 * 256 * 4 / (1024**2)  # MB
pair_memory = 256 * 256 * 128 * 4 / (1024**2)  # MB
extra_msa_memory = 5000 * 256 * 64 * 4 / (1024**2)  # MB
template_memory = 4 * 256 * 256 * 64 * 4 / (1024**2)  # MB (4 templates)

mem_labels = ['MSA\nrepr.', 'Pair\nrepr.', 'Extra\nMSA', 'Template\nfeatures']
mem_values = [msa_memory, pair_memory, extra_msa_memory, template_memory]
mem_colors = ['#85C1E9', '#F1948A', '#82E0AA', '#F9E79F']

bars3 = axes[2].bar(mem_labels, mem_values, color=mem_colors,
                     edgecolor='gray', linewidth=1.2)
for bar_item, val in zip(bars3, mem_values):
    axes[2].text(bar_item.get_x() + bar_item.get_width()/2, bar_item.get_height() + 2,
                 f'{val:.0f} MB', ha='center', fontsize=11, color='#2C3E50')
axes[2].set_ylabel('Memory (MB, float32)', fontsize=12)
axes[2].set_title('Approximate Memory Footprint\n(for $N_{res} = 256$)', fontsize=13)

plt.tight_layout()
plt.show()

print("=" * 70)
print("Key Takeaways:")
print("=" * 70)
print("1. Two core representations: MSA (N_seq x N_res x 256) and Pair (N_res x N_res x 128)")
print("2. Relative positional encoding is used instead of sinusoidal")
print("3. Templates inject structural priors into the pair representation")
print("4. Extra MSA sequences are processed in a separate lighter stack")
print("5. Recycling (3 iterations) progressively refines both representations")
print("=" * 70)
print("\nNext: Notebook 4 -- The Evoformer (Attention Mechanisms)")